# 1. Data cleansing

| | |
|---|---|
| **Aim** | Transform raw data into complete and clean dataset
| **Input data** | quaterly data `raw/2007Q1.csv` |
| **Result** | `clean/2007Q1.parquet` and reports in `reports/` |

1. **Random selection - 3% of loans** The entire data is too large. Random selection of loans, not rows, so that each loan has a complete history.
2. **Columns and types.** Dates become dates, and amounts become numbers. The 'XX' status becomes missing data.
3. **We remove impossible values.** For example, a DTI above 100 is treated as missing data.
4. **Limitation.** 30-year fixed-rate loans stay, without relief refinancing, and with complete key fields to ensure a homogeneous sample.
5. **Validation.** Each loan can only have one row per month. We also count gaps in the history.

We do not exclude loans repurchased by the seller, as repurchases are only known in the future.

In [1]:
import polars as pl

import config as C
import functions as F

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(70)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_hide_column_data_types(True)
C.REPORT_DIR.mkdir(parents=True, exist_ok=True)

## 1. File Processing

Files are processed sequentially.

In [2]:
C.CLEAN_DIR.mkdir(parents=True, exist_ok=True)
waterfall_rows, check_rows = [], []

for path in F.list_raw_files():
    cohort = path.stem
    df, n_all = F.load_cohort(path)
    df, waterfall = F.filter_population(df, cohort, n_all)
    df, check = F.check_panel(df, cohort)
    df.write_parquet(C.CLEAN_DIR / f"{cohort}.parquet")
    waterfall_rows += waterfall
    check_rows.append(check)
    print(f"{cohort}: {check['loans']} loans, {check['rows']} rows")

2007Q1: 6332 loans, 418924 rows


2019Q1: 8721 loans, 295850 rows


## 2. Exclusion

The table shows how many credits remain after each step.

In [3]:
waterfall = pl.DataFrame(waterfall_rows)
waterfall.write_csv(C.REPORT_DIR / "waterfall_by_cohort.csv")

total = waterfall.group_by("step").agg(pl.col("loans").sum(), pl.col("rows").sum()).sort("step")
total = total.with_columns(
    pct_of_previous=(pl.col("loans") / pl.col("loans").shift(1) * 100).round(1)
)
total.write_csv(C.REPORT_DIR / "waterfall.csv")
total

step,loans,rows,pct_of_previous
"""0. All loans in the file""",595144,0,null
"""1. Sampled loans""",17886,864846,3.0
"""2. Term of 360 months""",15053,714774,84.2
"""3. Product FRM""",15053,714774,100.0
"""4. No relief refinance""",15053,714774,100.0
"""5. Key fields present""",15053,714774,100.0


## 3. Validation

Duplicates should be 0.

In [4]:
checks = pl.DataFrame(check_rows)
checks.write_csv(C.REPORT_DIR / "panel_checks.csv")
checks

cohort,loans,rows,duplicates,loans_with_gaps,missing_months
"""2007Q1""",6332,418924,0,11,70
"""2019Q1""",8721,295850,0,0,0


## 4. Variables in clean dataset

In [5]:
pl.DataFrame(list(C.RAW_VARIABLES.items()), schema=["column", "description"], orient="row")

column,description
"""LOAN_ID""","""loan identifier"""
"""ACT_PERIOD""","""monthly reporting period"""
"""CSCORE_B""","""borrower FICO at origination"""
"""CSCORE_C""","""co-borrower FICO at origination"""
"""OLTV""","""LTV at origination (%)"""
"""OCLTV""","""CLTV at origination (%)"""
"""DTI""","""DTI at origination (%)"""
"""ORIG_RATE""","""interest rate at origination (%)"""
"""ORIG_UPB""","""loan amount (USD)"""
"""ORIG_TERM""","""loan term (months)"""
